# Data Preparation: election results (OpenCity)

**Input:** `interim/opencity_electiondata_{2014,2019,2024}.csv` from the data collection notebook.

**Output:** `clean/opencity_results_clean_2026-10-20.csv` (one row per real candidate per constituency per election), a data dictionary and a cleaning log 

**States:** Bihar, Maharashtra, Uttar Pradesh, Tamil Nadu (207 seats in each election).

**How the notebook is organised:** every cleaning step prints a count before and after, and writes one line to the cleaning log. In the log, `count_before` and `count_after` are counts of rows or of problem cells, as said in the `note` column.

In [1]:
# required imports
from pathlib import Path
import re
import difflib
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
# folders
project_dir = Path.cwd().parent
interim_dir = project_dir / "interim"
clean_dir = project_dir / "clean"
clean_dir.mkdir(parents=True, exist_ok=True)

COLLECTION_DATE = "2026-10-20"

In [3]:
# load the prepared datasets from the interim folder
data = {
    year: pd.read_csv(interim_dir / f"opencity_electiondata_{year}.csv", low_memory=False)
    for year in [2014, 2019, 2024]
}

# cleaning log: one line per step
# this a list of dictionaries
cleaning_log = []

def log_step(year, step, before, after, note=""):
    cleaning_log.append({
        "year": year, "step": step, "count_before": before,
        "count_after": after, "rows_affected": abs(before - after), "note": note
    })

### Step 1: keep the four states

The 2014 and 2019 files write state names with underscores (`Tamil_Nadu`), the 2024 file with spaces.

In [4]:
# keep only the states we want
states = ["Bihar", "Maharashtra", "Uttar Pradesh", "Tamil Nadu"]

# normalize column names
df_2014 = data[2014].rename(columns={"State_Name": "State"})
df_2019 = data[2019].rename(columns={"State_Name": "State"})
df_2024 = data[2024].rename(columns={"PC No": "Constituency_No", "PC Name": "Constituency_Name"})

# state names with an underscore, before and after
for year, df in [(2014, df_2014), (2019, df_2019)]:
    # how many states have an underscore
    before = df["State"].str.contains("_").sum()
    # replace underscores with a space
    df["State"] = df["State"].str.replace("_", " ")
    # this must now be zero
    after = df["State"].str.contains("_").sum()
    # log the step
    log_step(year, "state names: underscore to space", before, after, "Underscores in TCPD state names replaced with spaces to match ECI; counts = rows with '_' in State")
    print(year, "rows with underscore before:", before, "after:", after)

selected = {}
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    selected[year] = df[df["State"].isin(states)].copy()
    log_step(year, "keep four states", len(df), len(selected[year]), "Kept Bihar, Maharashtra, UP, Tamil Nadu; counts = rows"
)
    print(year, "rows before:", len(df), "after:", len(selected[year]))

df_2014, df_2019, df_2024 = selected[2014], selected[2019], selected[2024]

2014 rows with underscore before: 4010 after: 0
2019 rows with underscore before: 3447 after: 0
2014 rows before: 8843 after: 3868
2019 rows before: 8613 after: 3545
2024 rows before: 8902 after: 3626


### Step 2: remove by-elections

In the historical file `Poll_No` is 0 for the general election and 1 or more for by-elections. Four seat-years (two in 2014, two in 2019, among them Beed 2014 and Satara 2019) have vote shares that add up to 200, because the by-election was stored under the same constituency as the general election. Filtering on `Poll_No` fixes this and keeps all four seats. The 2024 file has no by-elections.

In [5]:
seat_keys = ["State", "Constituency_No"]

def seat_share_totals(frames):
    # sum of candidates' vote share per seat, should be about 100
    # .assign(Year=year) tags each total with its year
    return pd.concat([
        df.groupby(seat_keys)["Vote_Share_Percentage"].sum().rename("total").reset_index().assign(Year=year)
        for year, df in frames
    ])

# pd.concat stacks 2014 and 2019 into one table
totals_before = seat_share_totals([(2014, df_2014), (2019, df_2019)])
off = (totals_before["total"] - 100).abs() > 0.5
print("seats with total vote share far from 100 (before):", off.sum())
print(totals_before[off])

seats with total vote share far from 100 (before): 4
             State  Constituency_No   total  Year
78     Maharashtra               39  200.00  2014
147  Uttar Pradesh               21  199.99  2014
22           Bihar               23  199.99  2019
84     Maharashtra               45  200.00  2019


In [6]:
# keep only the general election
general = {year: df[df["Poll_No"] == 0].copy() for year, df in [(2014, df_2014), (2019, df_2019)]}
for year, df in [(2014, df_2014), (2019, df_2019)]:
    log_step(year, "drop by-election rows (Poll_No > 0)", len(df), len(general[year]),
             "By-election rows removed (Poll_No > 0); they made 4 seat-years' vote shares add up to ~200; counts = rows"
)
    print(year, "rows before:", len(df), "after:", len(general[year]))
df_2014, df_2019 = general[2014], general[2019]

totals_after = seat_share_totals([(2014, df_2014), (2019, df_2019)])
# check that this prints zero
print("seats with total vote share far from 100 (after):", ((totals_after["total"] - 100).abs() > 0.5).sum())

2014 rows before: 3868 after: 3844
2019 rows before: 3545 after: 3528
seats with total vote share far from 100 (after): 0


### Step 3: constituency names

**Matching rule:** the same seat in different years is matched by `State` + `Constituency_No`, never by name (names are spelled differently across sources). Names are only for reading. Parliamentary seats of these four states were not redrawn between 2014 and 2024, which the seat counts below confirm (Bihar 40, Maharashtra 48, Uttar Pradesh 80, Tamil Nadu 39).

In [7]:
# clean names + check that the seat counts are consistent across years (since they were not redrawn)

def clean_name(name):
    name = re.sub(r"\((SC|ST)\)", "", str(name), flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", name).strip().upper()

for df in [df_2014, df_2019, df_2024]:
    df["Constituency_Name"] = df["Constituency_Name"].map(clean_name)

# seat counts: exactly 207
expected = {"Bihar": 40, "Maharashtra": 48, "Uttar Pradesh": 80, "Tamil Nadu": 39}
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    # group rows by states and count the number of unique constituencies
    seats = df.groupby("State")["Constituency_No"].nunique().to_dict()
    # assert guards against mis-matches
    assert seats == expected, f"seat counts wrong in {year}"
print("seat counts OK in all three years:", expected)

seat counts OK in all three years: {'Bihar': 40, 'Maharashtra': 48, 'Uttar Pradesh': 80, 'Tamil Nadu': 39}


In [8]:
# the same seat is spelled differently in some years, we want to normalize these seat spellings

# drop_duplicates keeps one row per seat, set_index indexes using seat keys, we stack these with constituency names sharing common seat keys across different years on the same row
names = pd.concat({
    year: df.drop_duplicates(seat_keys).set_index(seat_keys)["Constituency_Name"]
    for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]
}, axis=1)

# flag the rows that have more than one unique value per row, these are the ones we need to normalize
different = names[names.nunique(axis=1) > 1]
print("seats with more than one spelling:", len(different))
print(different.to_string())

seats with more than one spelling: 9
                                                 2014                    2019                   2024
State         Constituency_No                                                                       
Bihar         31                          PATALIPUTRA             PATALIPUTRA             PATLIPUTRA
Maharashtra   11                   BHANDARA - GONDIYA      BHANDARA - GONDIYA       BHANDARA GONDIYA
              12                    GADCHIROLI-CHIMUR       GADCHIROLI-CHIMUR    GADCHIROLI - CHIMUR
              14                      YAVATMAL-WASHIM         YAVATMAL-WASHIM       YAVATMAL- WASHIM
              37                           AHMADNAGAR              AHMADNAGAR             AHMEDNAGAR
              46               RATNAGIRI - SINDHUDURG  RATNAGIRI - SINDHUDURG  RATNAGIRI- SINDHUDURG
              48                          HATKANANGLE             HATKANANGLE           HATKANANGALE
Tamil Nadu    1                           THIRUVALLUR 

In [9]:
# one spelling per seat: the 2019 spelling (TCPD), because two of three files agree most often (2014 and 2019)
canonical = names[2019]

pairs_before = pd.concat([df[["State", "Constituency_Name"]] for df in [df_2014, df_2019, df_2024]]).drop_duplicates().shape[0]
for df in [df_2014, df_2019, df_2024]:
    # map all the constituency names to a common name
    df["Constituency_Name"] = df.set_index(seat_keys).index.map(canonical).values

pairs_after = pd.concat([df[["State", "Constituency_Name"]] for df in [df_2014, df_2019, df_2024]]).drop_duplicates().shape[0]

log_step("all", "one name per seat", pairs_before, pairs_after, "2024 constituency spellings replaced with 2019 ones, counts = distinct (State, name) pairs over three years")
print("distinct (State, name) pairs before:", pairs_before, "after:", pairs_after)
assert pairs_after == 207

distinct (State, name) pairs before: 216 after: 207


### Step 4: numbers stored as text (2024)

In the 2024 file votes and vote share are text, and postal votes use `-` for zero. We check that EVM + postal = total on the full file (all states), then convert.

In [10]:
# note that this cell uses the whole data set and not the one with filtered states

raw_2024 = data[2024]

print("postal votes written as '-':", (raw_2024["Postal Votes"] == "-").sum())
print("EVM votes written as '-':", (raw_2024["EVM Votes"] == "-").sum())

# turns '-' into 0 and errors = 'coerce' turns anything that isnt already a number into Nan
votes = raw_2024[["EVM Votes", "Postal Votes", "Total Votes"]].replace("-", "0").apply(pd.to_numeric, errors="coerce")
complete = votes.notna().all(axis=1)
matches = votes["EVM Votes"] + votes["Postal Votes"] == votes["Total Votes"]

# verify claims
print("Rows checked (all states):", complete.sum())
print("Rows where EVM + Postal = Total:", (complete & matches).sum())
print("Rows where EVM + Postal != Total:", (complete & ~matches).sum())

postal votes written as '-': 578
EVM votes written as '-': 1
Rows checked (all states): 8902
Rows where EVM + Postal = Total: 8902
Rows where EVM + Postal != Total: 0


In [11]:
for col, new in [("Total Votes", "Votes"), ("Vote Share", "Vote_Share_Percentage")]:
    before = df_2024[col].map(lambda v: isinstance(v, str)).sum()
    df_2024[new] = pd.to_numeric(df_2024[col].replace("-", "0"))
    after = df_2024[new].map(lambda v: isinstance(v, str)).sum()
    log_step(2024, f"{col}: text to number", before, after,
             f"2024 df stores {col} as text; converted to numbers in {new}; counts = text cells")
    print(col, "text cells before:", before, "after:", after)
df_2024["Votes"] = df_2024["Votes"].astype(int)

# votes and vote shares: range and total checks for all years (should sum to 100)
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    share = df["Vote_Share_Percentage"]
    totals = df.groupby(seat_keys)["Vote_Share_Percentage"].sum()
    print(f"{year}: missing {share.isna().sum()}, below 0: {(share < 0).sum()}, above 100: {(share > 100.5).sum()}, "
          f"seat totals outside 99.5-100.5: {((totals - 100).abs() > 0.5).sum()}")

Total Votes text cells before: 3626 after: 0
Vote Share text cells before: 3626 after: 0
2014: missing 0, below 0: 0, above 100: 0, seat totals outside 99.5-100.5: 0
2019: missing 0, below 0: 0, above 100: 0, seat totals outside 99.5-100.5: 0
2024: missing 0, below 0: 0, above 100: 0, seat totals outside 99.5-100.5: 0


### Step 5: text inconsistencies in names and parties

Candidate names come in many forms: mixed case, dots (`KUMAR.P`), the `SURNAME, INITIALS` form used in Tamil Nadu (`Balaganapathy, V. Pon`), double spaces and titles (`DR`, `ADV`). Party names in 2024 have double spaces (`Janata Dal  (United)`).

**Rule:** upper case, dots and commas become spaces, other symbols removed, spaces collapsed, titles removed. The word order is kept for display. For matching, a separate key is made from the words sorted alphabetically, so `BALAGANAPATHY V PON` and `V PON BALAGANAPATHY` get the same key.

In [12]:
# put every candidate name in one format and build the matching keys

TITLES = {"DR", "ADV", "ADVOCATE", "SHRI", "SRI", "SMT", "SHRIMATI", "PROF", "PROFESSOR", "CAPT", "COL", "MR", "MRS", "MS", "SHREE"}

def clean_candidate(name):
    # 1. convert to uppercase, substiture all '.', ',' with spaces
    # 2. any other symbols are also replaced with spaces
    # 3. split breaks the name into words
    text = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper().replace(".", " ").replace(",", " "))
    tokens = [t for t in text.split() if t not in TITLES]
    return " ".join(tokens)

def name_key(name):
    # put the words in alphabetical order
    return " ".join(sorted(name.split()))

def name_problems(series):
    # counts five kinds of problems in a column, so we can track before and after counts
    s = series.astype(str)
    return {
        "not upper case": int((s != s.str.upper()).sum()),
        "has dot": int(s.str.contains(r"\.").sum()),
        "has comma": int(s.str.contains(",").sum()),
        "double space": int(s.str.contains(r"\s\s").sum()),
        "has title": int(s.map(lambda x: any(t in TITLES for t in re.sub(r"[^A-Za-z ]", " ", x).upper().split())).sum()),
    }

before_total = {}
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    problems_before = name_problems(df["Candidate"])
    df["Candidate"] = df["Candidate"].map(clean_candidate)
    df["Candidate_Key"] = df["Candidate"].map(name_key)
    problems_after = name_problems(df["Candidate"])

    for kind in problems_before:
        log_step(year, f"candidate names: {kind}", problems_before[kind], problems_after[kind],
         f"names upper-cased, dots/commas/symbols/titles removed; counts = names with '{kind}'")
    print(f"{year} before: {problems_before}")
    print(f"{year} after:  {problems_after}")

2014 before: {'not upper case': 208, 'has dot': 861, 'has comma': 186, 'double space': 76, 'has title': 159}
2014 after:  {'not upper case': 0, 'has dot': 0, 'has comma': 0, 'double space': 0, 'has title': 0}
2019 before: {'not upper case': 1632, 'has dot': 750, 'has comma': 160, 'double space': 0, 'has title': 168}
2019 after:  {'not upper case': 0, 'has dot': 0, 'has comma': 0, 'double space': 0, 'has title': 0}
2024 before: {'not upper case': 17, 'has dot': 448, 'has comma': 73, 'double space': 50, 'has title': 165}
2024 after:  {'not upper case': 0, 'has dot': 0, 'has comma': 0, 'double space': 0, 'has title': 0}


In [13]:
# collapse the double spaces in the 2024 party names
def tidy_spaces(series):
    return series.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

before = df_2024["Party"].str.contains(r"\s\s").sum()
df_2024["Party"] = tidy_spaces(df_2024["Party"])
after = df_2024["Party"].str.contains(r"\s\s").sum()
log_step(2024, "party names: double spaces", before, after, "repeated spaces in ECI party names collapsed; counts = party names with a double space")
print("party names with double spaces before:", before, "after:", after)

party names with double spaces before: 88 after: 0


### Step 6: NOTA, position and margin

`None of the Above` (NOTA) is stored as a candidate in all three files under different names. It cannot win, so it is flagged and left out of the ranking. Its votes stay in the seat total.

The published `Position` of 2014 and 2019 gives NOTA a rank like a candidate, which pushes real candidates one place down. Position and margin are therefore recomputed from votes, with the same rule for all three years.

In [14]:
# flag NOTA
# the files label NOTA rows using one of (NOTA / NONE OF THE ABOVE / None of the Above)

for df in [df_2014, df_2019, df_2024]:
    # add an indicator column that flags rows that are NOTA
    df["Is_NOTA"] = (
        df["Party"].isin(["NOTA", "None of the Above"])
        | df["Candidate"].isin(["NOTA", "NONE OF THE ABOVE"])
    )

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    print(year, "NOTA rows:", df["Is_NOTA"].sum(), "| names:", sorted(df.loc[df["Is_NOTA"], "Candidate"].unique()))

# 2019: one seat has no NOTA row
print("2019 seats without a NOTA row:", 207 - df_2019["Is_NOTA"].sum())

2014 NOTA rows: 207 | names: ['NONE OF THE ABOVE']
2019 NOTA rows: 206 | names: ['NOTA']
2024 NOTA rows: 207 | names: ['NOTA']
2019 seats without a NOTA row: 1


In [15]:
# re-rank candidates ignoring NOTA

position_fix = {}

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    # leave out the NOTA rows
    real_votes = df["Votes"].where(~df["Is_NOTA"])
    # ranks within each seat, with the most votes getting 1
    new_position = real_votes.groupby([df[k] for k in seat_keys]).rank(method="min", ascending=False)

    # wrong counts real candidates whose published position differs from the new one
    if "Position" in df.columns:
        wrong = ((df["Position"] != new_position) & ~df["Is_NOTA"]).sum()
    else:
        # 2024 has no published position
        wrong = 0
    position_fix[year] = wrong
    df["Position"] = new_position

    log_step(year, "position recomputed without NOTA", wrong, 0,
         "positions re-ranked by votes with NOTA excluded; counts = real candidates with a wrong published position")
    print(year, "real candidates with a wrong published position:", wrong, "of", (~df["Is_NOTA"]).sum())

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    winners = df[df["Position"] == 1].groupby(seat_keys).size()
    assert len(winners) == 207 and (winners == 1).all()
print("exactly one winner in every seat, every year")

2014 real candidates with a wrong published position: 2446 of 3637
2019 real candidates with a wrong published position: 2456 of 3322
2024 real candidates with a wrong published position: 0 of 3419
exactly one winner in every seat, every year


In [16]:
# add a winner flag, seat valid votes (includes NOTA votes), and winning margin (votes between 1st and 2nd)
def add_seat_columns(df):
    df = df.copy()
    # add the winner
    df["Winner"] = df["Position"] == 1
    # total vote count in a constituency
    df["Seat_Valid_Votes"] = df.groupby(seat_keys)["Votes"].transform("sum")

    # compute margin
    first = df[df["Position"] == 1].set_index(seat_keys)["Votes"]
    second = df[df["Position"] == 2].set_index(seat_keys)["Votes"]
    margin = (first - second).rename("Margin")

    # we drop the "Margin" column if it already exists, since we trust the margin we computed
    df = df.drop(columns=[c for c in ["Margin"] if c in df.columns])
    df = df.merge(margin, left_on=seat_keys, right_index=True, how="left")
    # counts the number of real candidates contesting per seat
    df["Seat_Candidates"] = df.groupby(seat_keys)["Is_NOTA"].transform(lambda s: (~s).sum())
    return df

df_2014, df_2019, df_2024 = [add_seat_columns(df) for df in [df_2014, df_2019, df_2024]]

for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    print(year, "margin min:", df["Margin"].min(), "| median:", df["Margin"].median(), "| missing:", df["Margin"].isna().sum())

2014 margin min: 1632.0 | median: 156756.0 | missing: 0
2019 margin min: 181.0 | median: 185720.0 | missing: 0
2024 margin min: 48 | median: 112482.0 | missing: 0


### Step 7: missing values

First the pattern: which columns are empty, in which years.

In [17]:
# columns that only the historical file has; 2024 does not have them at all
for col in ["Sex", "Constituency_Type", "Incumbent", "Turncoat", "Electors"]:
    df_2024[col] = np.nan
df_2024["Year"] = 2024

# check which columns are missing values, year-wise
check_cols = ["Sex", "Constituency_Type", "Incumbent", "Turncoat", "Electors", "Votes", "Vote_Share_Percentage", "Party"]
missing = pd.concat({
    year: df[check_cols].isna().sum()
    for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]
}, axis=1)

# add a row at the end, that contains each year's total row count
missing.loc["rows"] = [len(df_2014), len(df_2019), len(df_2024)]
missing

,2014,2019,2024
Sex,0,206,3626
Constituency_Type,0,0,3626
Incumbent,207,206,3626
Turncoat,207,206,3626
Electors,0,0,3626
Votes,0,0,0
Vote_Share_Percentage,0,0,0
Party,0,0,0
rows,3844,3528,3626


In [18]:
# 2014/2019: are the missing values only in NOTA rows? (NOTA rows = missing Incumbent/Turncoat exactly)
# we show that it indeed is 
historic = pd.concat([df_2014, df_2019], ignore_index=True)

for col in ["Sex", "Incumbent", "Turncoat"]:
    # contingency table: NOTA row (yes/no) x value missing (yes/no)
    table = pd.crosstab(historic["Is_NOTA"], historic[col].isna())
    # p is the chance of seeing such counts purely by chance
    chi2, p, _, _ = stats.chi2_contingency(table)
    print(f"\n{col}: missing vs NOTA row")
    print(table.to_string())
    print(f"chi-square = {chi2:.1f}, p = {p:.2e}")

# is any real candidate row missing a value?
real = historic[~historic["Is_NOTA"]]
print("\nmissing cells in real candidate rows (2014, 2019):")
print(real[["Sex", "Constituency_Type", "Incumbent", "Turncoat", "Electors"]].isna().sum().to_dict())

# There are 413 NOTA rows in total: 207 in 2014 and 206 in 2019. The two years store NOTA’s sex differently
# 2014: every NOTA row has the literal text "NOTA" in Sex. It’s filled in, so it isn’t counted as missing. Those are the 207 in the False column.
# 2019: every NOTA row has Sex empty (NaN). Those are the 206 in the True column.


Sex: missing vs NOTA row
Sex      False  True 
Is_NOTA              
False     6959      0
True       207    206
chi-square = 3552.5, p = 0.00e+00

Incumbent: missing vs NOTA row
Incumbent  False  True 
Is_NOTA                
False       6959      0
True           0    413
chi-square = 7353.1, p = 0.00e+00

Turncoat: missing vs NOTA row
Turncoat  False  True 
Is_NOTA               
False      6959      0
True          0    413
chi-square = 7353.1, p = 0.00e+00

missing cells in real candidate rows (2014, 2019):
{'Sex': 0, 'Constituency_Type': 0, 'Incumbent': 0, 'Turncoat': 0, 'Electors': 0}


**Mechanism.** 2014/2019: values are missing only in NOTA rows, where sex and incumbency do not apply (missing by definition). 2024: the 2024 file has no such columns (structural).

**Handling.** NOTA rows dropped (votes kept in seat totals). 2024 `Constituency_Type` copied from 2019, `Incumbent` derived in Step 10. `Sex`, `Turncoat`, `Electors` left missing for 2024.

In [19]:
# constituency type must be the same in 2014 and 2019 for every seat, otherwise copying it to 2024 is not safe
types = pd.concat([
    df.drop_duplicates(seat_keys).set_index(seat_keys)["Constituency_Type"].rename(year)
    for year, df in [(2014, df_2014), (2019, df_2019)]
], axis=1)
print("seats where type differs between 2014 and 2019:", (types[2014] != types[2019]).sum())
print("types in 2019:", types[2019].value_counts().to_dict())

before = df_2024["Constituency_Type"].isna().sum()
df_2024["Constituency_Type"] = df_2024.set_index(seat_keys).index.map(types[2019]).values
after = df_2024["Constituency_Type"].isna().sum()
log_step(2024, "constituency type imputed from 2019", before, after,
         "2024 file has no GEN/SC/ST; copied from 2019 by seat (unchanged 2014-19); counts = missing cells")
print("2024 missing constituency type before:", before, "after:", after)

seats where type differs between 2014 and 2019: 0
types in 2019: {'GEN': 168, 'SC': 35, 'ST': 4}
2024 missing constituency type before: 3626 after: 0


### Step 8: party codes

2014 and 2019 use short codes (BJP, INC). 2024 uses full names. The main parties of the four states are mapped to the short codes. Small parties in 2024 become `OTHER`, which is enough because the analysis only separates main parties, independents and the rest.

In [20]:
# give 2024 the same short codes as 2014 and 2019

party_map_2024 = {
    "Independent": "IND", "None of the Above": "NOTA",
    "Bharatiya Janata Party": "BJP", "Indian National Congress": "INC",
    "Bahujan Samaj Party": "BSP", "Samajwadi Party": "SP",
    "Rashtriya Janata Dal": "RJD", "Janata Dal (United)": "JD(U)",
    "Dravida Munnetra Kazhagam": "DMK",
    "All India Anna Dravida Munnetra Kazhagam": "ADMK",
    "Shiv Sena": "SHS", "Shiv Sena (Uddhav Balasaheb Thackrey)": "SHS(UBT)",
    "Nationalist Congress Party": "NCP",
    "Nationalist Congress Party – Sharadchandra Pawar": "NCP(SP)",
    "Naam Tamilar Katchi": "NTK", "Communist Party of India": "CPI",
    "Communist Party of India (Marxist)": "CPM",
    "Communist Party of India (Marxist-Leninist) (Liberation)": "CPI(ML)(L)",
    "Lok Janshakti Party(Ram Vilas)": "LJP(RV)", "Vanchit Bahujan Aaghadi": "VBA",
    "All India Majlis-E-Ittehadul Muslimeen": "AIMIM", "Pattali Makkal Katchi": "PMK",
    "Desiya Murpokku Dravida Kazhagam": "DMDK", "Viduthalai Chiruthaigal Katchi": "VCK",
    "Marumalarchi Dravida Munnetra Kazhagam": "MDMK", "Rashtriya Lok Dal": "RLD",
    "Hindustani Awam Morcha (Secular)": "HAMS",
}

# keep the original names too!
df_2024["Party_Full"] = df_2024["Party"]
# map to short forms
df_2024["Party"] = df_2024["Party_Full"].map(party_map_2024).fillna("OTHER")
log_step(2024, "party names to codes", len(df_2024), 0,
         f"ECI full names mapped to TCPD codes, {(df_2024['Party'] == 'OTHER').sum()} small-party rows set to OTHER; counts = rows with a full party name")
print("2024 rows mapped to OTHER:", (df_2024["Party"] == "OTHER").sum(), "of", len(df_2024))

for df in [df_2014, df_2019]:
    df["Party_Full"] = df["Party"]
    # different codes used for the same party, merge them into one
    df["Party"] = df["Party"].replace({"CPIM": "CPM"})
log_step("2014/2019", "CPIM merged into CPM", int(sum((d["Party_Full"] == "CPIM").sum() for d in [df_2014, df_2019])), 0,
         "TCPD uses two codes for CPI(M); counts = rows coded CPIM")

main_parties = ["BJP", "INC", "SP", "BSP", "RJD", "JD(U)", "DMK", "ADMK",
                "SHS", "SHS(UBT)", "NCP", "NCP(SP)"]   

# group each row: Main party, Independent, NOTA or Other party
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    df["Party_Group"] = np.select(
        [df["Party"].isin(main_parties), df["Party"] == "IND", df["Party"] == "NOTA"],
        ["Main party", "Independent", "NOTA"], default="Other party"
    )
    print(year, df["Party_Group"].value_counts().to_dict())

2024 rows mapped to OTHER: 983 of 3626
2014 {'Independent': 1503, 'Other party': 1302, 'Main party': 832, 'NOTA': 207}
2019 {'Independent': 1474, 'Other party': 1262, 'Main party': 586, 'NOTA': 206}
2024 {'Independent': 1698, 'Other party': 1116, 'Main party': 605, 'NOTA': 207}


### Step 9: duplicates

- **Exact duplicates:** identical rows. None found.
- **Namesakes:** two candidates with the same name in one seat and year. Usually different people (a namesake is fielded to split votes), so they are flagged, not merged.
- **Same person across years:** gets one `Person_ID`. Linked in the same seat by the same name (rule 1) or a near-identical name (rule 2), or in another seat of the same state by a unique name and the same party (rule 3). If an independent is involved, the name needs 3+ words. The links are tested against TCPD after matching.

In [21]:
# 1. exact duplicates: rows identical in every column (checked per year)
for year, df in [(2014, df_2014), (2019, df_2019), (2024, df_2024)]:
    exact = df.duplicated().sum()
    log_step(year, "exact duplicate rows", exact, 0, "rows identical in every column; none found")
    print(year, "exact duplicate rows:", exact)

# 2. stack the three years into one table with fresh row numbers
all_years = pd.concat([df_2014, df_2019, df_2024], ignore_index=True)

# 3. drop NOTA: it is not a candidate (its votes are already inside Seat_Valid_Votes)
real_rows = all_years[~all_years["Is_NOTA"]].copy()
log_step("all", "NOTA rows removed", len(all_years), len(real_rows), "NOTA is not a candidate; counts = rows")
print("\nrows before dropping NOTA:", len(all_years), "| after:", len(real_rows))

# 4. namesakes: same name key in the same seat and year; keep=False flags every copy
real_rows["Namesake_In_Seat"] = real_rows.duplicated(["Year", "State", "Constituency_No", "Candidate_Key"], keep=False)
print("rows that share a name with another candidate in the same seat:", real_rows["Namesake_In_Seat"].sum())
print(real_rows[real_rows["Namesake_In_Seat"]].groupby("Year").size().to_dict())

# 5. a few 2024 namesakes for display, sorted so each pair sits together
sample = real_rows[real_rows["Namesake_In_Seat"] & (real_rows["Year"] == 2024)]
print(sample.sort_values(["Constituency_Name", "Candidate_Key"])[["Constituency_Name", "Candidate", "Party", "Votes"]].head(8).to_string())

2014 exact duplicate rows: 0
2019 exact duplicate rows: 0
2024 exact duplicate rows: 0

rows before dropping NOTA: 10998 | after: 10378
rows that share a name with another candidate in the same seat: 58
{2014: 18, 2019: 18, 2024: 22}
      Constituency_Name           Candidate  Party     Votes
10229           ALIGARH        SATISH KUMAR    IND    3561.0
10230           ALIGARH        SATISH KUMAR  OTHER    1648.0
9173    CHENNAI CENTRAL     B PARTHASARATHY   DMDK   72016.0
9190    CHENNAI CENTRAL     B PARTHASARATHY    IND     380.0
9184    CHENNAI CENTRAL     S SATHISH KUMAR    IND     527.0
9186    CHENNAI CENTRAL     S SATHISH KUMAR    IND     501.0
7852          JAHANABAD  CHANDESHWAR PRASAD  JD(U)  300444.0
7860          JAHANABAD  CHANDESHWAR PRASAD    IND    5378.0


In [22]:
# the same politician appears in 2014, 2019 and 2024 as three separate rows
# we want to tag all three with one ID so we can follow that person over the years
# we use name, seat and party data to identify rows across the three years that represent the same person

# name words too common to count as evidence that two rows are one person
COMMON_WORDS = {"KUMAR", "SINGH", "PRASAD", "DEVI", "YADAV", "KUMARI", "SHARMA", "KHAN", "MOHAMMAD", "MOHD", "LAL", "KAUR",
                "NATH", "REDDY", "PATIL", "KUMARAN", "PANDEY", "VERMA", "GUPTA", "MISHRA", "SAHU", "PASWAN", "RAJ", "RAJA", "ALIAS"}

def allowed(nrow, orow):
    # nrow : the row from the newer election
    # orow : the row from the older election
    # namesake guard: if an independent is involved, a match needs a name of 3+ words
    weak = nrow["Party"] == "IND" or orow["Party"] == "IND"
    return (not weak) or len(nrow["Candidate_Key"].split()) >= 3

def match_years(old, new, used_old):
    """Link rows of a later election (new) to an earlier one (old), one-to-one.
    Returns a list of (old_index, new_index, rule)."""
    pairs = []          # results : (old row, new row, rule number)
    taken_new = set()   # new rows that already got matched
    # used_old (passed in) holds old rows that already got matched
    # together they make sure nobody is matched twice

    # ---------------- rule 1: same seat, exact same name ----------------
    # criteria: same state + same seat number + identical name key + passes the namesake guard
    for (state, seat), new_seat in new.groupby(seat_keys):
        # new_seat is the newer election's candidates in this seat
        # old_seat is the older election's candidates in the same seat that still haven't been matched
        old_seat = old[(old["State"] == state) & (old["Constituency_No"] == seat) & ~old.index.isin(used_old)]
        for key, new_rows in new_seat.groupby("Candidate_Key"):
            old_rows = old_seat[old_seat["Candidate_Key"] == key]          # old rows with this exact key
            # go through the new rows with this name, biggest vote-getter first (usually the real politician, not a look-alike)
            for i, nrow in new_rows.sort_values("Votes", ascending=False).iterrows():
                # decision: no old row left with this name -> nothing more to match for this key
                if old_rows.empty:
                    break
                # decision: keep only old rows that pass the namesake guard for THIS new row
                ok = old_rows[[allowed(nrow, orow) for _, orow in old_rows.iterrows()]]
                if ok.empty:
                    continue                                                # this new row can't be matched, try the next one
                # decision: if several old rows qualify, prefer the one from the same party
                same_party = ok[ok["Party"] == nrow["Party"]]
                pick = same_party.index[0] if len(same_party) else ok.index[0]
                # record the match and mark both rows as used
                pairs.append((pick, i, 1))
                taken_new.add(i)
                used_old = used_old | {pick}
                old_rows = old_rows.drop(pick)                              # each old row used once

    # ---------------- rule 2: same seat, similar spelling ----------------
    # criteria (either one is enough):
    #   a) spelling similarity >= 0.95 AND passes the namesake guard, or
    #   b) same real party (not IND/OTHER) AND at least 2 shared uncommon name words of 3+ letters
    for (state, seat), new_seat in new.groupby(seat_keys):
        old_seat = old[(old["State"] == state) & (old["Constituency_No"] == seat) & ~old.index.isin(used_old)]
        scores = []                                                         # every pair that qualifies, with a score
        # compare every unmatched new candidate with every unmatched old candidate in this seat
        for i, nrow in new_seat[~new_seat.index.isin(taken_new)].iterrows():
            new_words = set(nrow["Candidate_Key"].split())
            for j, orow in old_seat.iterrows():
                # SequenceMatcher: finds the longest run of characters the two names share,
                # then repeats on the leftover parts to the left and right of it, and adds up all matched characters (M).
                # ratio = 2 * M / (total characters in both names), so 1.0 = identical, 0 = nothing in common.
                # e.g. "GOPAL SHETTY" (12 chars) vs "CHINAYYA GOPAL SHETTY" (21 chars): M = 12 -> 2*12/33 = 0.73
                # more the ratio, stronger the chance that they are the same name spelled differently
                ratio = difflib.SequenceMatcher(None, nrow["Candidate_Key"], orow["Candidate_Key"]).ratio()
                shared = new_words & set(orow["Candidate_Key"].split())    # words found in both names
                strong_shared = sum(len(w) >= 3 and w not in COMMON_WORDS for w in shared) >= 2
                same_named_party = nrow["Party"] == orow["Party"] and nrow["Party"] not in ["IND", "OTHER"]
                # decision: does this pair qualify under (a) or (b)?
                if (ratio >= 0.95 and allowed(nrow, orow)) or (same_named_party and strong_shared):
                    scores.append((max(ratio, 0.9), j, i))                  # (b)-only pairs get at least 0.9 so they rank below (a)
        # decision: accept the best-scoring pairs first; skip a pair if either row is already taken
        for ratio, j, i in sorted(scores, reverse=True):
            if j in used_old or i in taken_new:
                continue
            pairs.append((j, i, 2))
            taken_new.add(i)
            used_old = used_old | {j}

    # ---------------- rule 3: same state, different seat (people who moved) ----------------
    # criteria: same state + identical name key (2+ words) + same party (not IND)
    #           + the name appears exactly once in the state in BOTH years, so there is no doubt who it is
    for state in new["State"].unique():
        # only unmatched, non-independent rows of this state
        old_state = old[(old["State"] == state) & ~old.index.isin(used_old) & (old["Party"] != "IND")]
        new_state = new[(new["State"] == state) & ~new.index.isin(taken_new) & (new["Party"] != "IND")]
        # how many times each name appears in the state, per year
        old_count = old_state["Candidate_Key"].value_counts()
        new_count = new_state["Candidate_Key"].value_counts()
        for i, nrow in new_state.iterrows():
            key = nrow["Candidate_Key"]
            # decision: skip short names, and names that are missing or repeated in either year
            if len(key.split()) < 2 or old_count.get(key, 0) != 1 or new_count.get(key, 0) != 1:
                continue
            # decision: the single old row with this name must also be from the same party
            match = old_state[(old_state["Candidate_Key"] == key) & (old_state["Party"] == nrow["Party"])]
            if len(match) == 1 and match.index[0] not in used_old:
                pairs.append((match.index[0], i, 3))
                taken_new.add(i)
                used_old = used_old | {match.index[0]}

    return pairs

In [23]:
# give a Person_ID to every real candidate row
# start: every row is its own person (ID = row number); matched rows then copy the older row's ID
real_rows = real_rows.reset_index(drop=True)          # row numbers 0..n, so index = starting ID
real_rows["Person_ID"] = np.arange(len(real_rows))
real_rows["Match_Rule"] = 0                           # 0 = not matched to an earlier year, else rule 1/2/3

def link(old_year, new_year):
    is_old = real_rows["Year"] == old_year
    is_new = real_rows["Year"] == new_year
    # a new row is "already linked" if its ID was changed by an earlier pass (ID no longer equals its row number)
    linked_new = is_new & (real_rows["Person_ID"] != real_rows.index)

    old = real_rows[is_old]
    new = real_rows[is_new & ~linked_new]             # only offer new rows that have no match yet

    # old rows whose person already shows up in the new year (through an earlier pass) are not offered again,
    # otherwise one person could end up with two rows in the same year
    already = set(real_rows.index[is_old & real_rows["Person_ID"].isin(real_rows.loc[linked_new, "Person_ID"])])

    pairs = match_years(old, new, already)            
    for j, i, rule in pairs:
        real_rows.loc[i, "Person_ID"] = real_rows.loc[j, "Person_ID"]   # new row takes the old row's ID
        real_rows.loc[i, "Match_Rule"] = rule
    return pairs

# order matters: 2019 rows get their 2014 IDs first, so 2024 then inherits the same ID through 2019
pairs_1419 = link(2014, 2019)
pairs_1924 = link(2019, 2024)
pairs_1424 = link(2014, 2024)                         # people who skipped 2019

# how many links each pass made, split by rule
for name, pairs in [("2014 -> 2019", pairs_1419), ("2019 -> 2024", pairs_1924), ("2014 -> 2024 (skipped 2019)", pairs_1424)]:
    print(name, "links:", len(pairs), "| by rule:", pd.Series([p[2] for p in pairs]).value_counts().sort_index().to_dict())

# how many elections each person contested (1, 2 or 3)
elections = real_rows.groupby("Person_ID")["Year"].nunique()
print("\npeople by number of elections contested (in these four states):", elections.value_counts().sort_index().to_dict())

# safety check: one person can't appear twice in the same election
assert real_rows.duplicated(["Person_ID", "Year"]).sum() == 0, "a person appears twice in one election"

2014 -> 2019 links: 322 | by rule: {1: 244, 2: 47, 3: 31}
2019 -> 2024 links: 326 | by rule: {1: 275, 2: 40, 3: 11}
2014 -> 2024 (skipped 2019) links: 54 | by rule: {1: 41, 2: 10, 3: 3}

people by number of elections contested (in these four states): {1: 9113, 2: 424, 3: 139}


In [24]:
# test the matching against TCPD's own 2019 columns
# Contested = number of elections TCPD says this person has ever stood in (anywhere in India)
# these columns came along with each row from the raw file, so no merge is needed
check = real_rows[real_rows["Year"] == 2019].copy()

# True if this 2019 person also has a 2014 row (same Person_ID)
check["linked_to_2014"] = check["Person_ID"].isin(real_rows.loc[real_rows["Year"] == 2014, "Person_ID"])

# accuracy: a link is certainly wrong if TCPD says 2019 was their first election
linked = check[check["linked_to_2014"]]
print("2019 candidates linked to a 2014 row by my rule:", len(linked))
print("   of these, TCPD says it was their first contest (a certainly wrong link):", (linked["Contested"] == 1).sum())
print(f"   precision (at least): {(linked['Contested'] > 1).mean():.3f}")

# coverage: of the people TCPD says stood before, how many did we link?
# many stood before in another state, which we cannot see, so this is expected to be well below 100%
has_earlier = check["Contested"] > 1
print("2019 candidates TCPD says contested before (anywhere in India):", has_earlier.sum())
print("   of these, linked to a 2014 row in the four states:", (has_earlier & check["linked_to_2014"]).sum())

2019 candidates linked to a 2014 row by my rule: 322
   of these, TCPD says it was their first contest (a certainly wrong link): 17
   precision (at least): 0.947
2019 candidates TCPD says contested before (anywhere in India): 535
   of these, linked to a 2014 row in the four states: 305


**Reading the test.** TCPD's `Contested` counts how many elections a person has contested, in any state. A link to an earlier election is wrong when it says 1 (first contest), so that count is an estimate of the error. The estimate is on the cautious side: TCPD's own linking of people is not perfect (for example, it gives `Contested` = 1 to the same Nanded candidate in 2019 who stood there for the same party in 2014), so some "wrong" links are right. Our files hold four states only, so some true re-contestants cannot be linked (they contested elsewhere before); that explains the missing recall and is accepted.

### Step 10: incumbent in 2024

An incumbent is the sitting MP. TCPD gives it for 2014 and 2019 only, so for 2024 we mark a candidate as incumbent if the same person won in 2019. We first test this rule on 2019 (using 2014 winners) against TCPD's own column. By-election winners are not counted (2 people in 2024).

In [25]:
# person ids of the winners in each year
winners = {y: set(real_rows.loc[(real_rows["Year"] == y) & real_rows["Winner"], "Person_ID"]) for y in [2014, 2019]}

# test the rule on 2019, where tcpd already gives the answer
test = real_rows[real_rows["Year"] == 2019]
derived = test["Person_ID"].isin(winners[2014])
published = test["Incumbent"].astype(str) == "True"
print(pd.crosstab(published, derived, rownames=["published"], colnames=["derived"]))
print(f"agreement: {(published == derived).mean():.4f}")

derived    False  True 
published              
False       3188      2
True          22    110
agreement: 0.9928


In [26]:
# 2024 has no incumbent column, so use the rule: won in 2019
mask_2024 = real_rows["Year"] == 2024
before = real_rows.loc[mask_2024, "Incumbent"].isna().sum()
real_rows.loc[mask_2024, "Incumbent"] = real_rows.loc[mask_2024, "Person_ID"].isin(winners[2019])
real_rows["Incumbent_Source"] = np.where(mask_2024, "derived", "published")

log_step(2024, "incumbent derived", before, 0, "2019 winners marked as sitting MPs; counts = empty cells")
print("2024 incumbents:", real_rows.loc[mask_2024, "Incumbent"].sum())

2024 incumbents: 122


### Step 11: outliers

Votes are very skewed (winners get 100 times the votes of independents), so a plain rule on all votes would flag every winner. Two checks are used instead.

**Candidate level:** `log10(Votes)`, compared inside a group of the same year and the same role (winner, runner-up, or other candidate). A candidate is flagged when the modified z-score (median and MAD) is beyond 3.5 in either direction.

**Seat level:** `Seat_Valid_Votes` and the winning margin as a share of the seat's votes, with the 1.5 x IQR rule within each year.

A flag only says "unusual". Each flagged value is then diagnosed with tests that would fail if a number were mistyped:
1. `Votes / Seat_Valid_Votes * 100` must equal the published vote share (within 0.05).
2. For 2014 and 2019, the published `Valid_Votes` of TCPD must equal the sum of the candidate votes in the seat.

In [27]:
def modified_z(series):
    median = series.median()
    mad = (series - median).abs().median()
    return 0.6745 * (series - median) / mad

votes_all = pd.concat([df_2014, df_2019, df_2024], ignore_index=True)
votes_all = votes_all[~votes_all["Is_NOTA"]].copy()
votes_all["log_votes"] = np.log10(votes_all["Votes"])
votes_all["role"] = np.select([votes_all["Position"] == 1, votes_all["Position"] == 2], ["winner", "runner-up"], default="other")
votes_all["z"] = votes_all.groupby(["Year", "role"])["log_votes"].transform(modified_z)
votes_all["Votes_Outlier"] = np.select([votes_all["z"] < -3.5, votes_all["z"] > 3.5], ["low", "high"], default="none")

print(pd.crosstab([votes_all["Year"], votes_all["role"]], votes_all["Votes_Outlier"]))

Votes_Outlier   high  low  none
Year role                      
2014 other        39    0  3184
     runner-up     0    0   207
     winner        0    0   207
2019 other        25    0  2883
     runner-up     0    3   204
     winner        1    1   205
2024 other        64    0  2941
     runner-up     0    8   199
     winner        0    0   207


In [28]:
# test 1: recompute vote share from votes and seat total, should match the published one
votes_all["share_error"] = (votes_all["Votes"] / votes_all["Seat_Valid_Votes"] * 100 - votes_all["Vote_Share_Percentage"]).abs()
flagged = votes_all[votes_all["Votes_Outlier"] != "none"]
print("all rows failing the share test:", (votes_all["share_error"] > 0.05).sum())
print("flagged rows:", len(flagged), "| failing the share test:", (flagged["share_error"] > 0.05).sum())

# test 2: our seat totals vs tcpd's published valid votes (2014, 2019)
for year, df in [(2014, df_2014), (2019, df_2019)]:
    seat = df.groupby(seat_keys).agg(published=("Valid_Votes", "first"), summed=("Votes", "sum"))
    print(year, "seats where the totals differ:", (seat["published"] != seat["summed"]).sum())

# a few examples of each kind of flag
show = ["Year", "Constituency_Name", "Candidate", "Party", "role", "Votes"]
print("\nlow flags:\n", flagged[flagged["Votes_Outlier"] == "low"].nsmallest(5, "z")[show].to_string())
print("\nhigh flags:\n", flagged[flagged["Votes_Outlier"] == "high"].nlargest(5, "z")[show].to_string())

all rows failing the share test: 0
flagged rows: 141 | failing the share test: 0
2014 seats where the totals differ: 0
2019 seats where the totals differ: 0

low flags:
       Year Constituency_Name               Candidate Party       role     Votes
9976  2024      THOOTHUKKUDI     SIVASAMY VELUMANI R  ADMK  runner-up  147991.0
9094  2024     CHENNAI NORTH               R MANOHAR  ADMK  runner-up  158111.0
9172  2024   CHENNAI CENTRAL                   VINOJ   BJP  runner-up  169159.0
9840  2024         THANJAVUR             SIVANESAN P  DMDK  runner-up  182662.0
5447  2019     CHENNAI NORTH  ALAGAAPURAM R MOHANRAJ  DMDK  runner-up  129468.0

high flags:
       Year Constituency_Name                   Candidate     Party   role     Votes
7485  2024        KISHANGANJ               AKHTARUL IMAN     AIMIM  other  309264.0
5357  2019            SANGLI  GOPICHAND KUNDLIK PADALKAR       VBA  other  300234.0
4003  2019        KISHANGANJ               AKHTARUL IMAN     AIMIM  other  295029.0


In [29]:
# seat level: seat size and winning margin as a share of the seat's votes
seats = votes_all.drop_duplicates(["Year", "State", "Constituency_No"])[["Year", "State", "Constituency_Name", "Seat_Valid_Votes", "Margin"]].copy()
seats["margin_pct"] = seats["Margin"] / seats["Seat_Valid_Votes"] * 100

for col in ["Seat_Valid_Votes", "margin_pct"]:
    seats[col + "_flag"] = False
    for year, part in seats.groupby("Year"):
        q1, q3 = part[col].quantile([0.25, 0.75])
        fence = 1.5 * (q3 - q1)
        seats.loc[part.index, col + "_flag"] = (part[col] < q1 - fence) | (part[col] > q3 + fence)

print("seats flagged for size:", seats["Seat_Valid_Votes_flag"].sum(), "| for margin:", seats["margin_pct_flag"].sum())
print("\nsmallest winning margins:")
print(seats.nsmallest(5, "Margin")[["Year", "State", "Constituency_Name", "Margin", "margin_pct"]].to_string())
print("\nlargest winning margins:")
print(seats.nlargest(5, "margin_pct")[["Year", "State", "Constituency_Name", "Margin", "margin_pct"]].to_string())
print("\nsmallest seats:")
print(seats.nsmallest(3, "Seat_Valid_Votes")[["Year", "State", "Constituency_Name", "Seat_Valid_Votes"]].to_string())
print("largest seats:")
print(seats.nlargest(3, "Seat_Valid_Votes")[["Year", "State", "Constituency_Name", "Seat_Valid_Votes"]].to_string())

seats flagged for size: 14 | for margin: 14

smallest winning margins:
      Year          State  Constituency_Name  Margin  margin_pct
8515  2024    Maharashtra  MUMBAI NORTH WEST    48.0    0.005026
7264  2019  Uttar Pradesh       MACHHLISHAHR   181.0    0.017489
933   2014    Maharashtra            HINGOLI  1632.0    0.155256
4448  2019          Bihar          JAHANABAD  1751.0    0.213000
1270  2014    Maharashtra             RAIGAD  2110.0    0.213613

largest winning margins:
      Year          State Constituency_Name    Margin  margin_pct
5446  2019     Tamil Nadu     CHENNAI NORTH  461518.0   48.298929
6533  2019  Uttar Pradesh    FATEHPUR SIKRI  495065.0   47.733165
3931  2019          Bihar         MADHUBANI  454940.0   47.207245
1148  2014    Maharashtra      MUMBAI NORTH  446582.0   47.179371
4969  2019    Maharashtra      MUMBAI NORTH  465247.0   47.006089

smallest seats:
      Year        State     Constituency_Name  Seat_Valid_Votes
9171  2024   Tamil Nadu       CHENNA

**Diagnosis.** Every flagged value passes both tests (vote share matches votes, seat totals match TCPD), so they are real, not typos. Low flags are runners-up in one-sided seats; high flags are mostly strong third-place candidates. Flagged seats are very large or small electorates, or very close or one-sided results (e.g. Mumbai North West 2024, won by 48 votes).

**Decision:** nothing is changed or removed. The candidate flag is kept in `Votes_Outlier` for plots.

In [30]:
log_step("all", "outliers flagged (candidate votes, MAD on log10 within role)", int((votes_all["Votes_Outlier"] != "none").sum()), 0,
         "rows flagged; all diagnosed as real values, none changed")
log_step("all", "outliers flagged (seat size or margin, IQR)", int((seats["Seat_Valid_Votes_flag"] | seats["margin_pct_flag"]).sum()), 0,
         "seat-years flagged; all diagnosed as real values, none changed")

### Step 12: the clean table

One row per real candidate per seat per election (NOTA removed). Columns kept:

| Column | Purpose |
|---|---|
| State | State of the seat |
| Constituency_No | Seat number in the state; with State, the key that identifies a seat across years |
| Constituency_Name | Seat name, one spelling per seat (for reading only) |
| Constituency_Type | GEN, SC or ST (reserved seat); 2024 copied from 2019 |
| Year | Election year: 2014, 2019 or 2024 |
| Person_ID | Same number for the same person across years |
| Candidate | Cleaned candidate name |
| Candidate_Key | Name words sorted A to Z, used to match people across years |
| Namesake_In_Seat | True if another candidate in the same seat and year has the same name |
| Party | Short party code (BJP, INC, ...); small 2024 parties are OTHER |
| Party_Full | Party name as published |
| Party_Group | Main party, Independent or Other party |
| Sex | M / F / O; empty for 2024 (not in the source) |
| Incumbent | True if the candidate was the sitting MP |
| Incumbent_Source | published (TCPD, 2014 and 2019) or derived (2024, from 2019 winners) |
| Position | Finishing place, recounted without NOTA |
| Winner | True for position 1 |
| Votes | Votes received |
| Vote_Share_Percentage | Share of all valid votes in the seat |
| Margin | Winner's votes minus runner-up's votes, same for every row of the seat |
| Seat_Valid_Votes | Total votes cast in the seat, NOTA included |
| Seat_Candidates | Number of real candidates in the seat |
| Votes_Outlier | low / high / none: unusual vote count for the candidate's role, checked and kept |

In [31]:
# same row order in both tables (checked), so the flag can be copied over by position
assert len(votes_all) == len(real_rows)
assert (votes_all["Candidate"].values == real_rows["Candidate"].values).all() and (votes_all["Votes"].values == real_rows["Votes"].values).all()
final = real_rows.copy()
final["Votes_Outlier"] = votes_all["Votes_Outlier"].values

final["Incumbent"] = final["Incumbent"].map(lambda v: np.nan if pd.isna(v) else str(v) == "True").astype("boolean")

cols = [
    "State", "Constituency_No", "Constituency_Name", "Constituency_Type", "Year",
    "Person_ID", "Candidate", "Candidate_Key", "Namesake_In_Seat",
    "Party", "Party_Full", "Party_Group", "Sex", "Incumbent", "Incumbent_Source",
    "Position", "Winner", "Votes", "Vote_Share_Percentage", "Margin",
    "Seat_Valid_Votes", "Seat_Candidates", "Votes_Outlier"
]
final = final[cols].sort_values(["Year", "State", "Constituency_No", "Position"]).reset_index(drop=True)

final["Position"] = final["Position"].astype("int64")
final["Votes"] = final["Votes"].astype("int64")
final["Margin"] = final["Margin"].astype("int64")
final["Seat_Valid_Votes"] = final["Seat_Valid_Votes"].astype("int64")
final["Seat_Candidates"] = final["Seat_Candidates"].astype("int64")

print(final.shape)
print(final.dtypes.to_string())

(10378, 23)
State                     object
Constituency_No            int64
Constituency_Name         object
Constituency_Type         object
Year                       int64
Person_ID                  int64
Candidate                 object
Candidate_Key             object
Namesake_In_Seat            bool
Party                     object
Party_Full                object
Party_Group               object
Sex                       object
Incumbent                boolean
Incumbent_Source          object
Position                   int64
Winner                      bool
Votes                      int64
Vote_Share_Percentage    float64
Margin                     int64
Seat_Valid_Votes           int64
Seat_Candidates            int64
Votes_Outlier             object


In [32]:
# final sanity checks
for year, df in final.groupby("Year"):
    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print("Rows:", len(df), "| seats:", df.groupby(["State", "Constituency_No"]).ngroups)
    print("Duplicate (person, year) rows:", df.duplicated(["Person_ID", "Year"]).sum())
    print("Missing in Candidate, Party, Votes, Vote_Share_Percentage:", df[["Candidate", "Party", "Votes", "Vote_Share_Percentage"]].isna().sum().sum())
    print("Negative votes:", (df["Votes"] < 0).sum())
    print("Winners per seat all 1:", (df.groupby(["State", "Constituency_No"])["Winner"].sum() == 1).all())
    print("Candidates per seat equals Seat_Candidates:", (df.groupby(["State", "Constituency_No"]).size().values == df.drop_duplicates(["State", "Constituency_No"])["Seat_Candidates"].values).all())


======== 2014 ========
Rows: 3637 | seats: 207
Duplicate (person, year) rows: 0
Missing in Candidate, Party, Votes, Vote_Share_Percentage: 0
Negative votes: 0
Winners per seat all 1: True
Candidates per seat equals Seat_Candidates: True

======== 2019 ========
Rows: 3322 | seats: 207
Duplicate (person, year) rows: 0
Missing in Candidate, Party, Votes, Vote_Share_Percentage: 0
Negative votes: 0
Winners per seat all 1: True
Candidates per seat equals Seat_Candidates: True

======== 2024 ========
Rows: 3419 | seats: 207
Duplicate (person, year) rows: 0
Missing in Candidate, Party, Votes, Vote_Share_Percentage: 0
Negative votes: 0
Winners per seat all 1: True
Candidates per seat equals Seat_Candidates: True


### Step 13: storage and data dictionary

**Folders:** `raw/` (downloads, never edited), `interim/` (year-wise extracts), `clean/` (final table, dictionary, log). File names carry source and date.

**Format: CSV.** About 10,400 rows and 23 columns, small enough that Parquet gives no real benefit, and CSV opens anywhere. CSV does not store column types, so the file is read back with a type map and checked below.

In [33]:
dictionary = pd.DataFrame([
    ("State", "str", "name", "OpenCity (TCPD / ECI)", "one of Bihar, Maharashtra, Uttar Pradesh, Tamil Nadu", "State of the constituency"),
    ("Constituency_No", "int", "number", "OpenCity (TCPD / ECI)", "1 to 80", "Parliamentary constituency number within the state"),
    ("Constituency_Name", "str", "name", "OpenCity (TCPD)", "207 distinct (State, name) pairs", "Constituency name, upper case, one spelling per seat"),
    ("Constituency_Type", "str", "category", "OpenCity (TCPD); 2024 copied from 2019", "GEN, SC, ST", "Reserved status of the seat"),
    ("Year", "int", "year", "OpenCity", "2014, 2019, 2024", "Election year (general election only)"),
    ("Person_ID", "int", "id", "derived (name matching)", ">= 0", "Same value for the same person across years (matching rule in Step 9)"),
    ("Candidate", "str", "name", "OpenCity (TCPD / ECI)", "non-empty", "Candidate name, upper case, no dots or titles"),
    ("Candidate_Key", "str", "name", "derived", "non-empty", "Words of the name in alphabetical order, used for matching"),
    ("Namesake_In_Seat", "bool", "flag", "derived", "True / False", "Another candidate in the same seat and year has the same Candidate_Key"),
    ("Party", "str", "code", "OpenCity; 2024 mapped by hand", "short code, OTHER for small 2024 parties", "Party code"),
    ("Party_Full", "str", "name", "OpenCity", "non-empty", "Party name as published (code in 2014 and 2019, full name in 2024)"),
    ("Party_Group", "str", "category", "derived", "Main party, Independent, Other party", "Main parties: BJP, INC, SP, BSP, RJD, JD(U), DMK, ADMK, SHS, SHS(UBT), NCP, NCP(SP)"),
    ("Sex", "str", "category", "OpenCity (TCPD); missing for 2024", "M, F, O, missing", "Sex of candidate"),
    ("Incumbent", "boolean", "flag", "OpenCity (TCPD); 2024 derived", "True / False", "Candidate was the sitting MP"),
    ("Incumbent_Source", "str", "category", "derived", "published, derived", "Where the Incumbent value comes from"),
    ("Position", "int", "rank", "derived from Votes", "1 to Seat_Candidates", "Finishing place among real candidates (NOTA not ranked)"),
    ("Winner", "bool", "flag", "derived", "True / False", "Position equals 1"),
    ("Votes", "int", "votes", "OpenCity (TCPD / ECI)", "100 to 1,200,000", "Votes received (EVM + postal)"),
    ("Vote_Share_Percentage", "float", "percent", "OpenCity (TCPD / ECI)", "0 to 100", "Share of all valid votes in the seat (NOTA included in the total)"),
    ("Margin", "int", "votes", "derived", ">= 1", "Votes of the winner minus votes of the runner-up, same for every row of a seat"),
    ("Seat_Valid_Votes", "int", "votes", "derived", "> 0", "Sum of votes of all rows in the seat, NOTA included"),
    ("Seat_Candidates", "int", "count", "derived", "2 to 60", "Number of real candidates in the seat"),
    ("Votes_Outlier", "str", "category", "derived (modified z-score on log10 votes within year and role)", "none, low, high", "Unusual vote count for the role, checked and kept"),
], columns=["column", "dtype", "unit", "source", "allowed_range", "description"])

dictionary

,column,dtype,unit,source,allowed_range,description
0,State,str,name,OpenCity (TCPD / ECI),"one of Bihar, Maharashtra, Uttar Pradesh, Tami...",State of the constituency
1,Constituency_No,int,number,OpenCity (TCPD / ECI),1 to 80,Parliamentary constituency number within the s...
2,Constituency_Name,str,name,OpenCity (TCPD),"207 distinct (State, name) pairs","Constituency name, upper case, one spelling pe..."
3,Constituency_Type,str,category,OpenCity (TCPD); 2024 copied from 2019,"GEN, SC, ST",Reserved status of the seat
4,Year,int,year,OpenCity,"2014, 2019, 2024",Election year (general election only)
5,Person_ID,int,id,derived (name matching),>= 0,Same value for the same person across years (m...
6,Candidate,str,name,OpenCity (TCPD / ECI),non-empty,"Candidate name, upper case, no dots or titles"
7,Candidate_Key,str,name,derived,non-empty,"Words of the name in alphabetical order, used ..."
8,Namesake_In_Seat,bool,flag,derived,True / False,Another candidate in the same seat and year ha...
9,Party,str,code,OpenCity; 2024 mapped by hand,"short code, OTHER for small 2024 parties",Party code


In [34]:
# check the allowed ranges that can be written as code
assert final["State"].isin(states).all()
assert final["Year"].isin([2014, 2019, 2024]).all()
assert final["Constituency_Type"].isin(["GEN", "SC", "ST"]).all()
assert final["Sex"].dropna().isin(["M", "F", "O"]).all()
assert final["Vote_Share_Percentage"].between(0, 100).all()
assert (final["Votes"] > 0).all() and (final["Margin"] >= 1).all()
assert final["Position"].between(1, final["Seat_Candidates"]).all()
assert final[["State", "Constituency_Name"]].drop_duplicates().shape[0] == 207
assert (final["Candidate"].str.len() > 0).all()
print("all allowed ranges hold")
print("Votes range:", final["Votes"].min(), "to", final["Votes"].max())
print("Seat_Candidates range:", final["Seat_Candidates"].min(), "to", final["Seat_Candidates"].max())

all allowed ranges hold
Votes range: 105 to 944503
Seat_Candidates range: 4 to 54


In [35]:
# save the clean table and the dictionary
clean_path = clean_dir / f"opencity_results_clean_{COLLECTION_DATE}.csv"
final.to_csv(clean_path, index=False)
dictionary.to_csv(clean_dir / f"opencity_results_data_dictionary_{COLLECTION_DATE}.csv", index=False)

# csv does not store types, so read it back with a type map and check nothing changed
dtype_map = {
    "State": "string", "Constituency_No": "int64", "Constituency_Name": "string", "Constituency_Type": "string",
    "Year": "int64", "Person_ID": "int64", "Candidate": "string", "Candidate_Key": "string", "Namesake_In_Seat": "bool",
    "Party": "string", "Party_Full": "string", "Party_Group": "string", "Sex": "string", "Incumbent": "boolean",
    "Incumbent_Source": "string", "Position": "int64", "Winner": "bool", "Votes": "int64",
    "Vote_Share_Percentage": "float64", "Margin": "int64", "Seat_Valid_Votes": "int64",
    "Seat_Candidates": "int64", "Votes_Outlier": "string",
}
back = pd.read_csv(clean_path, dtype=dtype_map)
print("shape equal:", back.shape == final.shape)
print("values equal:", back.astype(str).equals(final.astype(dtype_map).astype(str)))

shape equal: True
values equal: True


In [36]:
# save the cleaning log
clean_log = pd.DataFrame(cleaning_log)
clean_log.to_csv(clean_dir / f"opencity_results_cleaning_log_{COLLECTION_DATE}.csv", index=False)
print("saved:", sorted(p.name for p in clean_dir.glob("*.csv")))

saved: ['merged_all_2026-10-20.csv', 'merged_byelection_2026-10-20.csv', 'merged_general_2026-10-20.csv', 'merged_unmatched_myneta_2026-10-20.csv', 'opencity_results_clean_2026-10-20.csv', 'opencity_results_cleaning_log_2026-10-20.csv', 'opencity_results_data_dictionary_2026-10-20.csv', 'party_map_2026-10-20.csv']


### Cleaning log

In [37]:
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
clean_log

,year,step,count_before,count_after,rows_affected,note
0,2014,state names: underscore to space,4010,0,4010,Underscores in TCPD state names replaced with spaces to match ECI; counts = ...
1,2019,state names: underscore to space,3447,0,3447,Underscores in TCPD state names replaced with spaces to match ECI; counts = ...
2,2014,keep four states,8843,3868,4975,"Kept Bihar, Maharashtra, UP, Tamil Nadu; counts = rows"
3,2019,keep four states,8613,3545,5068,"Kept Bihar, Maharashtra, UP, Tamil Nadu; counts = rows"
4,2024,keep four states,8902,3626,5276,"Kept Bihar, Maharashtra, UP, Tamil Nadu; counts = rows"
5,2014,drop by-election rows (Poll_No > 0),3868,3844,24,By-election rows removed (Poll_No > 0); they made 4 seat-years' vote shares ...
6,2019,drop by-election rows (Poll_No > 0),3545,3528,17,By-election rows removed (Poll_No > 0); they made 4 seat-years' vote shares ...
7,all,one name per seat,216,207,9,"2024 constituency spellings replaced with 2019 ones, counts = distinct (Stat..."
8,2024,Total Votes: text to number,3626,0,3626,2024 df stores Total Votes as text; converted to numbers in Votes; counts = ...
9,2024,Vote Share: text to number,3626,0,3626,2024 df stores Vote Share as text; converted to numbers in Vote_Share_Percen...
